# Applying the Fraud Prediction Pipeline to a New Dataset

This notebook walks through two scenarios:

1. **Inference only** — you have a trained model and want to score new transactions
2. **Retrain** — you have new labeled data and want to train a fresh model

---

## What your data needs to look like

The pipeline is flexible. Your dataset needs:

| Requirement | Details |
|---|---|
| A binary fraud label | Any column name works — you'll specify it in config |
| Numeric or categorical features | Categoricals get label-encoded automatically |
| No missing values in the label column | Feature NaNs are dropped row-wise |
| CSV or Parquet format | Loaded via `src/data/ingest.py` |

**Don't have a dataset yet?** This notebook generates a realistic synthetic one so you can follow along immediately.

## Setup

In [ ]:
# Run this from the repo root: fraud-prediction-pipeline/
import sys, os
# Make sure we're at the repo root so src/ imports work
if 'notebooks' in os.getcwd():
    os.chdir('..')
print('Working directory:', os.getcwd())

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
print('Ready.')

---
## Step 1 — Bring Your Own Data

Replace the cell below with a path to your CSV or Parquet file.  
We generate a synthetic dataset here so every cell runs out of the box.

In [ ]:
# ── Option A: load your own file ──────────────────────────────────────────────
# YOUR_DATA_PATH = 'data/raw/my_transactions.csv'
# df_raw = pd.read_csv(YOUR_DATA_PATH)
# YOUR_TARGET = 'is_fraud'           # <-- change to your label column name
# YOUR_CATEGORICALS = ['merchant_category', 'card_type']   # <-- your cat cols
# YOUR_NUMERICS = ['amount', 'distance_from_home']         # <-- your num cols

# ── Option B: generate synthetic data (default, runs immediately) ─────────────
from sklearn.datasets import make_classification

rng = np.random.default_rng(42)
n = 5_000

X_syn, y_syn = make_classification(
    n_samples=n, n_features=15, n_informative=8, n_redundant=3,
    weights=[0.96, 0.04],   # ~4% fraud — realistic imbalance
    flip_y=0.01, random_state=42,
)

feature_names = (
    ['amount', 'distance_from_home', 'distance_from_last_txn',
     'ratio_to_median', 'used_chip', 'used_pin', 'online_order']
    + [f'behavioral_{i}' for i in range(8)]
)

df_raw = pd.DataFrame(X_syn, columns=feature_names)
df_raw['merchant_category'] = rng.choice(['grocery','gas','online','travel','restaurant'], n)
df_raw['card_type']         = rng.choice(['visa','mastercard','amex'], n)
df_raw['amount']            = np.abs(df_raw['amount']) * 100   # make realistic
df_raw['is_fraud']          = y_syn

YOUR_TARGET       = 'is_fraud'
YOUR_CATEGORICALS = ['merchant_category', 'card_type']
YOUR_NUMERICS     = [c for c in df_raw.columns if c not in YOUR_CATEGORICALS + [YOUR_TARGET]]

df_raw.to_csv('data/raw/synthetic_transactions.csv', index=False)
print(f'Dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

### 1a — Understand your class balance

Fraud detection is almost always a severe imbalance problem. Check it before anything else.

In [ ]:
counts = df_raw[YOUR_TARGET].value_counts()
fraud_rate = df_raw[YOUR_TARGET].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Class counts
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

# Amount distribution by class
for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    subset = df_raw[df_raw[YOUR_TARGET] == label]['amount']
    axes[1].hist(subset.clip(upper=subset.quantile(0.99)), bins=50,
                 alpha=0.6, color=color, label='Legit' if label == 0 else 'Fraud')
axes[1].set_title('Transaction Amount by Class')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.savefig('reports/eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Fraud rate: {fraud_rate*100:.2f}%')
if fraud_rate < 0.01:
    print('⚠  Very severe imbalance (<1%). SMOTE and PR-AUC are especially important here.')
elif fraud_rate < 0.05:
    print('Moderate imbalance. SMOTE will help.')
else:
    print('Mild imbalance. Model should handle this reasonably without SMOTE.')

---
## Step 2 — Preprocess

The pipeline modules in `src/data/preprocess.py` handle everything. We call them directly here so you can see each transformation.

In [ ]:
from src.data.preprocess import clean, encode_categoricals, split, scale_numerics, apply_smote

# 1. Clean
df_clean = clean(df_raw)
print(f'Rows after dedup/dropna: {len(df_clean):,}  (removed {len(df_raw)-len(df_clean):,})')

# 2. Encode categoricals
df_enc, encoders = encode_categoricals(df_clean, YOUR_CATEGORICALS)
print(f'Encoded: {YOUR_CATEGORICALS}')

# 3. Stratified split  (train 70 / val 10 / test 20)
train, val, test = split(df_enc, target=YOUR_TARGET, test_size=0.2, val_size=0.1)
print(f'\nSplit → train={len(train):,}  val={len(val):,}  test={len(test):,}')
print(f'Fraud rate  train={train[YOUR_TARGET].mean()*100:.2f}%  '
      f'val={val[YOUR_TARGET].mean()*100:.2f}%  '
      f'test={test[YOUR_TARGET].mean()*100:.2f}%')

In [ ]:
# 4. Scale numerics (fit on train only — prevents leakage)
X_train = train.drop(columns=[YOUR_TARGET])
y_train = train[YOUR_TARGET]
X_val   = val.drop(columns=[YOUR_TARGET])
y_val   = val[YOUR_TARGET]
X_test  = test.drop(columns=[YOUR_TARGET])
y_test  = test[YOUR_TARGET]

X_train_sc, scaler = scale_numerics(X_train, YOUR_NUMERICS, fit=True)
X_val_sc,   _      = scale_numerics(X_val,   YOUR_NUMERICS, scaler=scaler, fit=False)
X_test_sc,  _      = scale_numerics(X_test,  YOUR_NUMERICS, scaler=scaler, fit=False)

print('Scaler fit on train only — mean/std from test set never seen during fitting.')

# 5. SMOTE on train only
X_train_res, y_train_res = apply_smote(X_train_sc, y_train, sampling_strategy=0.1)
print(f'After SMOTE: {len(X_train_res):,} rows  fraud rate={y_train_res.mean()*100:.1f}%')

---
## Step 3 — Feature Engineering

> **Adapting to your data:** `src/features/engineering.py` contains velocity/time features designed for timestamped transaction data. If your dataset has a timestamp column, pass it in. If not (like our synthetic data), skip those and add your own domain-specific features below.

In [ ]:
def add_domain_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Add features here that make sense for YOUR domain.
    Examples below work for the synthetic dataset.
    For real transaction data, see src/features/engineering.py
    for velocity windows and time-of-day features.
    """
    X = X.copy()

    # Log-transform skewed amount
    if 'amount' in X.columns:
        X['amount_log1p'] = np.log1p(np.abs(X['amount']))

    # Ratio: distance travelled vs typical spend pattern
    if 'distance_from_home' in X.columns and 'distance_from_last_txn' in X.columns:
        X['distance_ratio'] = (
            X['distance_from_home'] / (X['distance_from_last_txn'].abs() + 1e-9)
        )

    # Add your own features here:
    # X['my_feature'] = ...

    return X

X_train_feat = add_domain_features(X_train_res)
X_val_feat   = add_domain_features(X_val_sc)
X_test_feat  = add_domain_features(X_test_sc)

FEATURE_NAMES = list(X_train_feat.columns)
print(f'Final feature count: {len(FEATURE_NAMES)}')
print('Features:', FEATURE_NAMES)

---
## Step 4 — Train with Hyperparameter Search

We use the same `src/models/train.py` module. Trials and folds are kept small here — scale them up for production.

In [ ]:
from src.models.train import tune_and_train, save_model

# Tune XGBoost (increase n_trials to 30-50 for production)
model, best_params, cv_score = tune_and_train(
    X_train_feat, y_train_res,
    model_type='xgboost',
    n_trials=5,       # ← increase for production
    cv_folds=3,       # ← increase for production
    seed=42,
    experiment_name='new-dataset-walkthrough',
)

print(f'\nBest CV PR-AUC : {cv_score:.4f}')
print(f'Best params    : {best_params}')

---
## Step 5 — Evaluate

Always evaluate on the **held-out test set** — never the data used for training or cross-validation.

In [ ]:
from src.evaluation.metrics import evaluate, find_best_threshold, print_report
from sklearn.metrics import precision_recall_curve, roc_curve, auc

metrics = evaluate(model, X_test_feat, y_test)
print_report(metrics)

proba = model.predict_proba(X_test_feat)[:, 1]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Precision-Recall curve
prec_c, rec_c, _ = precision_recall_curve(y_test, proba)
axes[0].plot(rec_c, prec_c, color='tomato', lw=2,
             label=f'PR-AUC = {metrics["pr_auc"]:.3f}')
axes[0].axhline(y_test.mean(), linestyle='--', color='grey', label='Baseline (random)')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve'); axes[0].legend()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'ROC-AUC = {metrics["roc_auc"]:.3f}')
axes[1].plot([0,1],[0,1], linestyle='--', color='grey', label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend()

# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay
preds = (proba >= metrics['threshold']).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, preds, display_labels=['Legit','Fraud'],
    colorbar=False, ax=axes[2], cmap='Blues'
)
axes[2].set_title(f'Confusion Matrix (threshold={metrics["threshold"]:.2f})')

plt.tight_layout()
Path('reports').mkdir(exist_ok=True)
plt.savefig('reports/evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(fi)*0.3)))
fi.tail(15).plot.barh(ax=ax, color='steelblue')
ax.set_title('Top Feature Importances (XGBoost)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('reports/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 6 — Save the Model

Artifacts are saved to `models/`. The serving API (`src/serving/app.py`) loads them automatically.

In [ ]:
import joblib, json

save_model(model, FEATURE_NAMES, output_dir='models')
joblib.dump(scaler,   'models/scaler.pkl')
joblib.dump(encoders, 'models/encoders.pkl')
json.dump({'threshold': metrics['threshold']}, open('models/threshold.json','w'))

print('Saved:')
for f in Path('models').iterdir():
    print(f'  {f}  ({f.stat().st_size/1024:.1f} KB)')

---
## Step 7 — Score New Transactions (Inference)

Once a model is saved you can load it at any time and score new data — no retraining needed.  
This is the path for **daily/real-time scoring** on incoming transactions.

In [ ]:
# Simulate 10 brand-new transactions arriving
new_txns = df_raw.drop(columns=[YOUR_TARGET]).sample(10, random_state=99)
true_labels = df_raw.loc[new_txns.index, YOUR_TARGET].values

# ── Inference pipeline ───────────────────────────────────────────────────────
# 1. Load artifacts
loaded_model    = joblib.load('models/best_model.pkl')
loaded_scaler   = joblib.load('models/scaler.pkl')
loaded_encoders = joblib.load('models/encoders.pkl')
loaded_features = json.load(open('models/feature_names.json'))
loaded_threshold = json.load(open('models/threshold.json'))['threshold']

# 2. Encode categoricals (use fitted encoders — never refit on new data)
new_enc, _ = encode_categoricals(new_txns, YOUR_CATEGORICALS,
                                  encoders=loaded_encoders, fit=False)

# 3. Scale (use fitted scaler)
new_sc, _ = scale_numerics(new_enc, YOUR_NUMERICS, scaler=loaded_scaler, fit=False)

# 4. Feature engineering (same function as training)
new_feat = add_domain_features(new_sc)

# 5. Align columns (guard against column order or missing feature drift)
new_feat = new_feat.reindex(columns=loaded_features, fill_value=0)

# 6. Score
fraud_proba = loaded_model.predict_proba(new_feat)[:, 1]
predictions = (fraud_proba >= loaded_threshold).astype(int)

# Display results
results = pd.DataFrame({
    'true_label':  ['FRAUD' if l else 'legit' for l in true_labels],
    'fraud_prob':  fraud_proba.round(4),
    'prediction':  ['FRAUD' if p else 'legit' for p in predictions],
    'correct':     ['✓' if p == t else '✗' for p, t in zip(predictions, true_labels)],
})
results.style.apply(
    lambda row: ['background-color: #fdd' if row['correct'] == '✗' else '' for _ in row],
    axis=1
)

---
## Step 8 — REST API (Production Serving)

For real-time scoring at scale, use the FastAPI endpoint instead of loading the model in Python directly.

```bash
# Start the server
uvicorn src.serving.app:app --reload
```

Then POST a transaction:

```bash
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "transaction_id": "txn_001",
    "features": {
      "amount": 450.0,
      "distance_from_home": 8.7,
      ...
    }
  }'
```

Response:
```json
{
  "transaction_id": "txn_001",
  "fraud_probability": 0.847,
  "is_fraud": true,
  "threshold": 0.5
}
```

---
## Adapting to Your Schema — Checklist

| Task | Where to change |
|---|---|
| Different label column name | `YOUR_TARGET` variable above, `configs/pipeline.yaml → data.target_column` |
| Different categorical columns | `YOUR_CATEGORICALS`, `configs/pipeline.yaml → preprocessing.categorical_columns` |
| Different numeric columns | `YOUR_NUMERICS`, `configs/pipeline.yaml → preprocessing.numeric_columns` |
| Add velocity/time features | `src/features/engineering.py → add_velocity_features()` — pass your timestamp column |
| Add domain features | Extend `add_domain_features()` in this notebook, then copy into `src/features/engineering.py` |
| Change model type | `tune_and_train(..., model_type='lightgbm')` |
| Change imbalance strategy | Swap `apply_smote` for `RandomOverSampler`, `ADASYN`, or class weights |
| Run full CLI pipeline | `python scripts/run_pipeline.py --config configs/my_dataset.yaml` |

---
*Dataset used in this walkthrough: synthetic — generated with `sklearn.datasets.make_classification`. See `scripts/run_creditcard.py` for the real ULB credit card fraud example.*